# Notebook 4: Machine Learning Model Training
## AI-Powered Smart University Assistant

This notebook covers:
- Training 3 models: Logistic Regression, Random Forest, XGBoost
- 5-Fold Stratified Cross-Validation
- Hyperparameter tuning (GridSearchCV) for RF and XGBoost
- Model comparison
- Final model selection and saving

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle, os, sys
sys.path.append('..')
warnings = __import__('warnings'); warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print('All libraries loaded.')

## 1. Load and Prepare Data

In [ ]:
df = pd.read_csv('../data/processed/student_features.csv')

target_col = 'At_Risk'
drop_cols = [c for c in ['id_student', 'code_module', 'code_presentation', 'final_result', target_col] if c in df.columns]
X = df.drop(columns=drop_cols)
y = df[target_col]

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X.select_dtypes(include=['number']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Training set: {X_train.shape}, Test set: {X_test.shape}')
print(f'At-Risk in train: {y_train.mean()*100:.1f}%')

## 2. Build Pipeline

In [ ]:
def build_pipeline(classifier):
    preprocessor = ColumnTransformer(transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])
    return Pipeline(steps=[('preprocessor', preprocessor), ('classifier', classifier)])

## 3. Train Base Models + Cross-Validation

In [ ]:
scale_pos = len(y_train[y_train==0]) / len(y_train[y_train==1])

base_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, scale_pos_weight=scale_pos)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, clf in base_models.items():
    print(f'\nTraining {name}...')
    pipe = build_pipeline(clf)
    pipe.fit(X_train, y_train)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'  CV F1 Scores: {scores.round(4)}')
    print(f'  Mean: {scores.mean():.4f} (+/- {scores.std():.4f})')

## 4. Hyperparameter Tuning (GridSearchCV)

In [ ]:
# Tune Random Forest
print('Tuning Random Forest...')
rf_params = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}
rf_grid = GridSearchCV(
    build_pipeline(RandomForestClassifier(random_state=42, class_weight='balanced')),
    rf_params, cv=StratifiedKFold(3), scoring='f1', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)
print(f'Best RF Params: {rf_grid.best_params_}')
print(f'Best RF F1: {rf_grid.best_score_:.4f}')

In [ ]:
# Tune XGBoost
print('Tuning XGBoost...')
xgb_params = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 6],
    'classifier__learning_rate': [0.05, 0.1]
}
xgb_grid = GridSearchCV(
    build_pipeline(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, scale_pos_weight=scale_pos)),
    xgb_params, cv=StratifiedKFold(3), scoring='f1', n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train, y_train)
print(f'Best XGB Params: {xgb_grid.best_params_}')
print(f'Best XGB F1: {xgb_grid.best_score_:.4f}')

## 5. Model Comparison

In [ ]:
def evaluate(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4)
    }

# Build and fit base LR for comparison
lr_pipe = build_pipeline(LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
lr_pipe.fit(X_train, y_train)

results_list = [
    evaluate('Logistic Regression', lr_pipe, X_test, y_test),
    evaluate('Random Forest (Tuned)', rf_grid.best_estimator_, X_test, y_test),
    evaluate('XGBoost (Tuned)', xgb_grid.best_estimator_, X_test, y_test)
]
results_df = pd.DataFrame(results_list).set_index('Model')
print('\n=== Model Comparison ===')
print(results_df.to_string())

In [ ]:
# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.15
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
colors = ['#6366f1', '#10b981', '#f59e0b', '#ef4444', '#38bdf8']
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, alpha=0.85)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df.index, rotation=10)
ax.set_ylim(0, 1.0)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../screenshots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Best Model

In [ ]:
# Select best model by F1
best_idx = results_df['F1'].idxmax()
print(f'Best model: {best_idx}')

best_model = {
    'Logistic Regression': lr_pipe,
    'Random Forest (Tuned)': rf_grid.best_estimator_,
    'XGBoost (Tuned)': xgb_grid.best_estimator_
}[best_idx]

os.makedirs('../models', exist_ok=True)
with open('../models/risk_prediction_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

results_dict = results_df.T.to_dict()
with open('../models/model_results.pkl', 'wb') as f:
    pickle.dump(results_dict, f)

print('Best model and results saved to models/!')

## Why Recall Matters More Than Accuracy

In academic risk prediction, **Recall (Sensitivity)** for the At-Risk class is more important than overall accuracy.

- A **False Negative** (predicting Not-At-Risk when student IS at risk) means a struggling student receives no intervention — potentially leading to course failure or withdrawal.
- A **False Positive** (predicting At-Risk when student is NOT at risk) results in an unnecessary but harmless check-in with an advisor.

Therefore, we prioritize **high Recall** over accuracy when selecting the final model, and use `class_weight='balanced'` to reduce false negatives.